# NOTEBOOK 08C
## Receiver-Level Signal and Model Discrimination

Objectives:
1. Evaluate performance within each outer fold.
2. Compare models against the training-mean baseline.
3. Aggregate predictions at receiver level.
4. Evaluate receiver-level discrimination.
5. Perform receiver-cluster bootstrap.
6. Investigate error concentration.
7. Examine draft-cohort performance.

This notebook does not refit models or modify the frozen dataset.


In [1]:

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OOF_PATH = ROOT / "reports/nested_ridge_modeling/oof_predictions.csv"
DATASET_PATH = ROOT / "data/features/wr_analysis_dataset_core_common.parquet"

OUTPUT_DIR = ROOT / "reports/receiver_level_signal"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOOTSTRAP_REPLICATES = 5000
RANDOM_SEED = 2027

print("Project root:", ROOT)
print("OOF exists:", OOF_PATH.exists())
print("Dataset exists:", DATASET_PATH.exists())


Project root: D:\PROJECTS\WR-MovementIQ
OOF exists: True
Dataset exists: True


In [2]:

if not OOF_PATH.exists():
    raise FileNotFoundError(OOF_PATH)

oof = pd.read_csv(OOF_PATH)

required = [
    "outcome", "model", "nfl_id", "season",
    "draft_year", "outer_fold", "actual", "predicted"
]

missing = set(required) - set(oof.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

for col in [
    "actual", "predicted", "season",
    "draft_year", "outer_fold"
]:
    oof[col] = pd.to_numeric(oof[col], errors="coerce")

oof["nfl_id"] = oof["nfl_id"].astype(str)

if oof[required].isna().any().any():
    raise ValueError("Missing values in required OOF fields.")

print("Rows:", len(oof))
print("Receivers:", oof.nfl_id.nunique())
print("Outcomes:", sorted(oof.outcome.unique()))
print("Models:", sorted(oof.model.unique()))
print("Folds:", sorted(oof.outer_fold.unique()))


Rows: 2232
Receivers: 54
Outcomes: ['game_mean_acceleration_raw_median', 'game_mean_speed_raw_median', 'game_p95_speed_raw_median', 'game_path_efficiency_median', 'game_peak_acceleration_raw_median', 'game_peak_speed_raw_median']
Models: ['BASELINE', 'COMBINED', 'TRACKING', 'TRADITIONAL']
Folds: [1, 2, 3, 4, 5]


In [3]:

EXPECTED_MODELS = {
    "BASELINE", "TRADITIONAL",
    "TRACKING", "COMBINED"
}

if set(oof.model.unique()) != EXPECTED_MODELS:
    raise ValueError("Unexpected model set.")

if oof.duplicated(
    ["outcome", "model", "nfl_id", "season", "draft_year"]
).any():
    raise ValueError("Duplicate OOF keys detected.")

coverage = (
    oof.groupby(["outcome", "model"])
    .agg(
        observations=("nfl_id", "size"),
        receivers=("nfl_id", "nunique")
    )
    .reset_index()
)

display(coverage)

for outcome in oof.outcome.unique():

    samples = {}

    for model in EXPECTED_MODELS:

        part = oof[
            (oof.outcome == outcome) &
            (oof.model == model)
        ]

        samples[model] = set(
            zip(part.nfl_id, part.season, part.draft_year)
        )

    reference = samples["BASELINE"]

    for model in EXPECTED_MODELS:

        if samples[model] != reference:
            raise ValueError(
                f"Sample mismatch: {outcome}, {model}"
            )

print("PASS: identical player-season samples.")


,outcome,model,observations,receivers
0,game_mean_acceleration_raw_median,BASELINE,93,54
1,game_mean_acceleration_raw_median,COMBINED,93,54
2,game_mean_acceleration_raw_median,TRACKING,93,54
3,game_mean_acceleration_raw_median,TRADITIONAL,93,54
4,game_mean_speed_raw_median,BASELINE,93,54
5,game_mean_speed_raw_median,COMBINED,93,54
6,game_mean_speed_raw_median,TRACKING,93,54
7,game_mean_speed_raw_median,TRADITIONAL,93,54
8,game_p95_speed_raw_median,BASELINE,93,54
9,game_p95_speed_raw_median,COMBINED,93,54


PASS: identical player-season samples.


In [4]:

def safe_corr(x, y, method="pearson"):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 3:
        return np.nan

    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan

    if method == "pearson":
        return float(pearsonr(x, y).statistic)

    return float(spearmanr(x, y).statistic)


## 1. Outer-fold performance

In [5]:

records = []

for (outcome, model, fold), part in oof.groupby(
    ["outcome", "model", "outer_fold"]
):

    actual = part.actual.to_numpy(float)
    predicted = part.predicted.to_numpy(float)

    actual_sd = np.std(actual, ddof=1)
    prediction_sd = np.std(predicted, ddof=1)

    slope = (
        np.cov(actual, predicted, ddof=1)[0, 1]
        / np.var(actual, ddof=1)
        if actual_sd > 1e-12 else np.nan
    )

    records.append({
        "outcome": outcome,
        "model": model,
        "outer_fold": int(fold),
        "observations": len(part),
        "receivers": part.nfl_id.nunique(),
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted)),
        "r2": r2_score(actual, predicted),
        "pearson_r": safe_corr(actual, predicted),
        "spearman_rho": safe_corr(actual, predicted, "spearman"),
        "prediction_sd_ratio": (
            prediction_sd / actual_sd
            if actual_sd > 1e-12 else np.nan
        ),
        "calibration_slope": slope
    })

fold_metrics = pd.DataFrame(records)

baseline = (
    fold_metrics[fold_metrics.model == "BASELINE"]
    [["outcome", "outer_fold", "mae", "rmse"]]
    .rename(columns={
        "mae": "baseline_mae",
        "rmse": "baseline_rmse"
    })
)

fold_metrics = fold_metrics.merge(
    baseline,
    on=["outcome", "outer_fold"],
    how="left"
)

fold_metrics["mae_difference_vs_baseline"] = (
    fold_metrics.mae - fold_metrics.baseline_mae
)

fold_metrics["rmse_difference_vs_baseline"] = (
    fold_metrics.rmse - fold_metrics.baseline_rmse
)

fold_metrics.to_csv(
    OUTPUT_DIR / "fold_level_model_metrics.csv",
    index=False
)

display(fold_metrics.sort_values(
    ["outcome", "model", "outer_fold"]
))


,outcome,model,outer_fold,observations,receivers,mae,rmse,r2,pearson_r,spearman_rho,prediction_sd_ratio,calibration_slope,baseline_mae,baseline_rmse,mae_difference_vs_baseline,rmse_difference_vs_baseline
0,game_mean_acceleration_raw_median,BASELINE,1,19,11,0.119617,0.146566,-0.001278,NaN,NaN,0.000000e+00,0.000000e+00,0.119617,0.146566,0.000000,0.000000
1,game_mean_acceleration_raw_median,BASELINE,2,19,11,0.145776,0.180816,-0.150474,NaN,NaN,2.634347e-15,-2.191510e-30,0.145776,0.180816,0.000000,0.000000
2,game_mean_acceleration_raw_median,BASELINE,3,19,11,0.124204,0.159898,-0.007854,NaN,NaN,2.788221e-15,4.910008e-30,0.124204,0.159898,0.000000,0.000000
3,game_mean_acceleration_raw_median,BASELINE,4,18,10,0.130080,0.177581,-0.658461,NaN,NaN,0.000000e+00,0.000000e+00,0.130080,0.177581,0.000000,0.000000
4,game_mean_acceleration_raw_median,BASELINE,5,18,11,0.205103,0.242413,-0.001900,NaN,NaN,0.000000e+00,0.000000e+00,0.205103,0.242413,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,game_peak_speed_raw_median,TRADITIONAL,1,19,11,0.454992,0.554768,-0.585864,0.456988,0.231894,6.580119e-01,3.007037e-01,0.442237,0.524614,0.012755,0.030155
116,game_peak_speed_raw_median,TRADITIONAL,2,19,11,0.388800,0.466359,-0.414762,0.188501,0.135804,7.486866e-01,1.411279e-01,0.346316,0.395531,0.042484,0.070828
117,game_peak_speed_raw_median,TRADITIONAL,3,19,11,0.416153,0.583988,-0.054998,0.241138,0.257171,5.209963e-01,1.256320e-01,0.431316,0.655157,-0.015163,-0.071169
118,game_peak_speed_raw_median,TRADITIONAL,4,18,10,0.769317,0.957638,-0.146904,0.784080,0.816215,6.213728e-02,4.872058e-02,0.822222,1.029420,-0.052906,-0.071782


## 2. Receiver-level aggregation

In [6]:

receiver_predictions = (
    oof.groupby(["outcome", "model", "nfl_id"], as_index=False)
    .agg(
        actual_receiver_mean=("actual", "mean"),
        predicted_receiver_mean=("predicted", "mean"),
        seasons=("season", "nunique"),
        player_seasons=("season", "size"),
        draft_year=("draft_year", "min")
    )
)

receiver_predictions.to_csv(
    OUTPUT_DIR / "receiver_level_predictions.csv",
    index=False
)

records = []

for (outcome, model), part in receiver_predictions.groupby(
    ["outcome", "model"]
):

    actual = part.actual_receiver_mean.to_numpy(float)
    predicted = part.predicted_receiver_mean.to_numpy(float)

    actual_sd = np.std(actual, ddof=1)
    prediction_sd = np.std(predicted, ddof=1)

    records.append({
        "outcome": outcome,
        "model": model,
        "receivers": len(part),
        "pearson_r": safe_corr(actual, predicted),
        "spearman_rho": safe_corr(actual, predicted, "spearman"),
        "receiver_mean_mae": mean_absolute_error(actual, predicted),
        "receiver_mean_rmse": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "receiver_mean_r2": r2_score(actual, predicted),
        "prediction_sd_ratio": (
            prediction_sd / actual_sd
            if actual_sd > 1e-12 else np.nan
        )
    })

receiver_metrics = pd.DataFrame(records)

receiver_metrics.to_csv(
    OUTPUT_DIR / "receiver_level_discrimination.csv",
    index=False
)

display(receiver_metrics.sort_values(["outcome", "model"]))


,outcome,model,receivers,pearson_r,spearman_rho,receiver_mean_mae,receiver_mean_rmse,receiver_mean_r2,prediction_sd_ratio
0,game_mean_acceleration_raw_median,BASELINE,54,-0.260137,-0.273542,0.134197,0.169157,-0.040464,0.066216
1,game_mean_acceleration_raw_median,COMBINED,54,-0.173473,-0.190928,0.143323,0.177433,-0.144765,0.235380
2,game_mean_acceleration_raw_median,TRACKING,54,-0.186264,-0.281342,0.144011,0.176874,-0.137558,0.222949
3,game_mean_acceleration_raw_median,TRADITIONAL,54,-0.237142,-0.103335,0.138532,0.175128,-0.115211,0.176862
4,game_mean_speed_raw_median,BASELINE,54,-0.433965,-0.445478,0.344366,0.463922,-0.092440,0.081847
5,game_mean_speed_raw_median,COMBINED,54,-0.243482,-0.255498,0.361343,0.479126,-0.165218,0.230273
6,game_mean_speed_raw_median,TRACKING,54,-0.297427,-0.317477,0.370238,0.489937,-0.218398,0.255544
7,game_mean_speed_raw_median,TRADITIONAL,54,-0.224470,-0.240709,0.360666,0.472864,-0.134961,0.204842
8,game_p95_speed_raw_median,BASELINE,54,-0.390776,-0.395311,0.433899,0.582132,-0.135835,0.135846
9,game_p95_speed_raw_median,COMBINED,54,0.273568,0.335468,0.428610,0.544407,0.006610,0.529202


## 3. Receiver-level errors

In [7]:

errors = oof.copy()

errors["absolute_error"] = (
    errors.actual - errors.predicted
).abs()

receiver_error = (
    errors.groupby(
        ["outcome", "model", "nfl_id"],
        as_index=False
    )
    .agg(receiver_mae=("absolute_error", "mean"))
)

display(receiver_error.head())


,outcome,model,nfl_id,receiver_mae
0,game_mean_acceleration_raw_median,BASELINE,55887,0.175042
1,game_mean_acceleration_raw_median,BASELINE,55904,0.102956
2,game_mean_acceleration_raw_median,BASELINE,55915,0.063248
3,game_mean_acceleration_raw_median,BASELINE,55920,0.242969
4,game_mean_acceleration_raw_median,BASELINE,55928,0.384147


## 4. Receiver-cluster bootstrap

In [8]:

def bootstrap_receiver_mae(
    outcome, model, rng, n_boot=BOOTSTRAP_REPLICATES
):

    part = receiver_error[
        (receiver_error.outcome == outcome) &
        (receiver_error.model == model)
    ]

    values = part.receiver_mae.to_numpy(float)
    n = len(values)

    if n < 5:
        return np.nan, np.nan, np.nan, n

    boot = np.empty(n_boot)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = values[idx].mean()

    return (
        values.mean(),
        np.quantile(boot, 0.025),
        np.quantile(boot, 0.975),
        n
    )


def bootstrap_paired_difference(
    outcome, model, rng, n_boot=BOOTSTRAP_REPLICATES
):

    m = receiver_error[
        (receiver_error.outcome == outcome) &
        (receiver_error.model == model)
    ][["nfl_id", "receiver_mae"]].rename(
        columns={"receiver_mae": "model_mae"}
    )

    b = receiver_error[
        (receiver_error.outcome == outcome) &
        (receiver_error.model == "BASELINE")
    ][["nfl_id", "receiver_mae"]].rename(
        columns={"receiver_mae": "baseline_mae"}
    )

    paired = m.merge(
        b, on="nfl_id", validate="one_to_one"
    )

    differences = (
        paired.model_mae - paired.baseline_mae
    ).to_numpy(float)

    n = len(differences)

    if n < 5:
        return np.nan, np.nan, np.nan, n

    boot = np.empty(n_boot)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = differences[idx].mean()

    return (
        differences.mean(),
        np.quantile(boot, 0.025),
        np.quantile(boot, 0.975),
        n
    )


rng = np.random.default_rng(RANDOM_SEED)

records = []

for outcome in sorted(oof.outcome.unique()):

    for model in sorted(EXPECTED_MODELS):

        estimate, low, high, n = bootstrap_receiver_mae(
            outcome, model, rng
        )

        diff, diff_low, diff_high, n_pair = (
            bootstrap_paired_difference(
                outcome, model, rng
            )
        )

        records.append({
            "outcome": outcome,
            "model": model,
            "receivers": n,
            "receiver_equal_weight_mae": estimate,
            "mae_ci_lower": low,
            "mae_ci_upper": high,
            "mae_difference_vs_baseline": diff,
            "difference_ci_lower": diff_low,
            "difference_ci_upper": diff_high,
            "paired_receivers": n_pair
        })

bootstrap_results = pd.DataFrame(records)

bootstrap_results.to_csv(
    OUTPUT_DIR / "receiver_cluster_bootstrap.csv",
    index=False
)

display(bootstrap_results.sort_values(["outcome", "model"]))


,outcome,model,receivers,receiver_equal_weight_mae,mae_ci_lower,mae_ci_upper,mae_difference_vs_baseline,difference_ci_lower,difference_ci_upper,paired_receivers
0,game_mean_acceleration_raw_median,BASELINE,54,0.146812,0.122457,0.172459,0.000000,0.000000,0.000000,54
1,game_mean_acceleration_raw_median,COMBINED,54,0.152827,0.126311,0.180074,0.006015,-0.002568,0.016065,54
2,game_mean_acceleration_raw_median,TRACKING,54,0.152872,0.126905,0.179180,0.006061,-0.000870,0.013943,54
3,game_mean_acceleration_raw_median,TRADITIONAL,54,0.151204,0.124405,0.177975,0.004392,-0.001286,0.010538,54
4,game_mean_speed_raw_median,BASELINE,54,0.380429,0.301149,0.464836,0.000000,0.000000,0.000000,54
5,game_mean_speed_raw_median,COMBINED,54,0.397635,0.318331,0.482095,0.017206,-0.002384,0.037772,54
6,game_mean_speed_raw_median,TRACKING,54,0.406031,0.325193,0.492098,0.025602,0.007278,0.045161,54
7,game_mean_speed_raw_median,TRADITIONAL,54,0.396385,0.318385,0.475929,0.015956,-0.001668,0.033105,54
8,game_p95_speed_raw_median,BASELINE,54,0.463281,0.369174,0.569806,0.000000,0.000000,0.000000,54
9,game_p95_speed_raw_median,COMBINED,54,0.464636,0.384505,0.557076,0.001355,-0.056158,0.058751,54


## 5. Error concentration

In [9]:

records = []

for (outcome, model), part in receiver_error.groupby(
    ["outcome", "model"]
):

    ranked = part.sort_values(
        "receiver_mae",
        ascending=False
    )

    total = ranked.receiver_mae.sum()

    records.append({
        "outcome": outcome,
        "model": model,
        "receivers": len(ranked),
        "top5_receiver_error_share": (
            ranked.head(5).receiver_mae.sum() / total
            if total > 0 else np.nan
        ),
        "top10_receiver_error_share": (
            ranked.head(10).receiver_mae.sum() / total
            if total > 0 else np.nan
        ),
        "worst_receiver_mae": ranked.receiver_mae.max(),
        "median_receiver_mae": ranked.receiver_mae.median()
    })

error_concentration = pd.DataFrame(records)

error_concentration.to_csv(
    OUTPUT_DIR / "receiver_error_concentration.csv",
    index=False
)

display(error_concentration)


,outcome,model,receivers,top5_receiver_error_share,top10_receiver_error_share,worst_receiver_mae,median_receiver_mae
0,game_mean_acceleration_raw_median,BASELINE,54,0.213692,0.373100,0.384147,0.145112
1,game_mean_acceleration_raw_median,COMBINED,54,0.212384,0.374877,0.382681,0.142381
2,game_mean_acceleration_raw_median,TRACKING,54,0.212465,0.369751,0.382708,0.148463
3,game_mean_acceleration_raw_median,TRADITIONAL,54,0.214426,0.383123,0.382682,0.145215
4,game_mean_speed_raw_median,BASELINE,54,0.260881,0.449832,1.225597,0.279280
5,game_mean_speed_raw_median,COMBINED,54,0.250760,0.433064,1.213787,0.314343
6,game_mean_speed_raw_median,TRACKING,54,0.254185,0.434144,1.213822,0.312069
7,game_mean_speed_raw_median,TRADITIONAL,54,0.249452,0.430675,1.213795,0.335269
8,game_p95_speed_raw_median,BASELINE,54,0.286813,0.450855,1.729500,0.352292
9,game_p95_speed_raw_median,COMBINED,54,0.244999,0.401703,1.531632,0.374137


## 6. Draft-cohort descriptive metrics

In [10]:

cohort_predictions = (
    oof.groupby(
        ["outcome", "model", "nfl_id", "draft_year"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        player_seasons=("season", "size")
    )
)

records = []

for (outcome, model, cohort), part in cohort_predictions.groupby(
    ["outcome", "model", "draft_year"]
):

    actual = part.actual.to_numpy(float)
    predicted = part.predicted.to_numpy(float)

    records.append({
        "outcome": outcome,
        "model": model,
        "draft_year": int(cohort),
        "receivers": part.nfl_id.nunique(),
        "pearson_r": safe_corr(actual, predicted),
        "spearman_rho": safe_corr(actual, predicted, "spearman"),
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted))
    })

cohort_metrics = pd.DataFrame(records)

cohort_metrics.to_csv(
    OUTPUT_DIR / "draft_cohort_receiver_metrics.csv",
    index=False
)

display(cohort_metrics.sort_values(
    ["outcome", "model", "draft_year"]
))


,outcome,model,draft_year,receivers,pearson_r,spearman_rho,mae,rmse
0,game_mean_acceleration_raw_median,BASELINE,2023,20,-0.241348,-0.295580,0.143501,0.185455
1,game_mean_acceleration_raw_median,BASELINE,2024,20,-0.347181,-0.334441,0.115741,0.140859
2,game_mean_acceleration_raw_median,BASELINE,2025,14,-0.166617,-0.282246,0.147271,0.181358
3,game_mean_acceleration_raw_median,COMBINED,2023,20,-0.138962,-0.300752,0.164415,0.201065
4,game_mean_acceleration_raw_median,COMBINED,2024,20,-0.318127,-0.360902,0.119509,0.151873
...,...,...,...,...,...,...,...,...
67,game_peak_speed_raw_median,TRACKING,2024,20,-0.246603,-0.287218,0.479918,0.665431
68,game_peak_speed_raw_median,TRACKING,2025,14,-0.100833,-0.305495,0.430505,0.496334
69,game_peak_speed_raw_median,TRADITIONAL,2023,20,0.328691,0.532331,0.373904,0.499882
70,game_peak_speed_raw_median,TRADITIONAL,2024,20,-0.085375,-0.264662,0.543471,0.680962


## 7. Model versus baseline decision table

In [11]:

comparison = bootstrap_results[
    bootstrap_results.model != "BASELINE"
].copy()

comparison["bootstrap_conclusion"] = np.select(
    [
        comparison.difference_ci_upper < 0,
        comparison.difference_ci_lower > 0
    ],
    [
        "IMPROVEMENT",
        "WORSE_THAN_BASELINE"
    ],
    default="UNCERTAIN"
)

comparison.to_csv(
    OUTPUT_DIR / "model_vs_baseline_decisions.csv",
    index=False
)

display(comparison[[
    "outcome",
    "model",
    "mae_difference_vs_baseline",
    "difference_ci_lower",
    "difference_ci_upper",
    "bootstrap_conclusion"
]])


,outcome,model,mae_difference_vs_baseline,difference_ci_lower,difference_ci_upper,bootstrap_conclusion
1,game_mean_acceleration_raw_median,COMBINED,0.006015,-0.002568,0.016065,UNCERTAIN
2,game_mean_acceleration_raw_median,TRACKING,0.006061,-0.000870,0.013943,UNCERTAIN
3,game_mean_acceleration_raw_median,TRADITIONAL,0.004392,-0.001286,0.010538,UNCERTAIN
5,game_mean_speed_raw_median,COMBINED,0.017206,-0.002384,0.037772,UNCERTAIN
6,game_mean_speed_raw_median,TRACKING,0.025602,0.007278,0.045161,WORSE_THAN_BASELINE
7,game_mean_speed_raw_median,TRADITIONAL,0.015956,-0.001668,0.033105,UNCERTAIN
9,game_p95_speed_raw_median,COMBINED,0.001355,-0.056158,0.058751,UNCERTAIN
10,game_p95_speed_raw_median,TRACKING,0.019688,-0.012434,0.052266,UNCERTAIN
11,game_p95_speed_raw_median,TRADITIONAL,0.011099,-0.033110,0.056926,UNCERTAIN
13,game_path_efficiency_median,COMBINED,0.000176,-0.000413,0.000864,UNCERTAIN


## 8. Research integrity manifest

In [12]:

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = {
    "notebook": "08C_receiver_level_signal_and_model_discrimination",
    "oof_file": str(OOF_PATH.relative_to(ROOT)),
    "dataset_file": str(DATASET_PATH.relative_to(ROOT)),
    "dataset_sha256": (
        sha256_file(DATASET_PATH)
        if DATASET_PATH.exists() else None
    ),
    "oof_rows": int(len(oof)),
    "receivers": int(oof.nfl_id.nunique()),
    "outcomes": sorted(oof.outcome.unique().tolist()),
    "models": sorted(oof.model.unique().tolist()),
    "outer_folds": sorted(
        oof.outer_fold.astype(int).unique().tolist()
    ),
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "bootstrap_seed": RANDOM_SEED,
    "bootstrap_unit": "nfl_id",
    "models_refitted": False,
    "dataset_modified": False
}

with open(
    OUTPUT_DIR / "receiver_level_signal_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=4)

print(json.dumps(manifest, indent=4))


{
    "notebook": "08C_receiver_level_signal_and_model_discrimination",
    "oof_file": "reports\\nested_ridge_modeling\\oof_predictions.csv",
    "dataset_file": "data\\features\\wr_analysis_dataset_core_common.parquet",
    "dataset_sha256": "02fbd802522973f7c4d100da57cd680f7bbfde6346a0431a7aa1fe5e25807c5c",
    "oof_rows": 2232,
    "receivers": 54,
    "outcomes": [
        "game_mean_acceleration_raw_median",
        "game_mean_speed_raw_median",
        "game_p95_speed_raw_median",
        "game_path_efficiency_median",
        "game_peak_acceleration_raw_median",
        "game_peak_speed_raw_median"
    ],
    "models": [
        "BASELINE",
        "COMBINED",
        "TRACKING",
        "TRADITIONAL"
    ],
    "outer_folds": [
        1,
        2,
        3,
        4,
        5
    ],
    "bootstrap_replicates": 5000,
    "bootstrap_seed": 2027,
    "bootstrap_unit": "nfl_id",
    "models_refitted": false,
    "dataset_modified": false
}


## 9. Final output audit

In [13]:

expected_outputs = [
    "fold_level_model_metrics.csv",
    "receiver_level_predictions.csv",
    "receiver_level_discrimination.csv",
    "receiver_cluster_bootstrap.csv",
    "receiver_error_concentration.csv",
    "draft_cohort_receiver_metrics.csv",
    "model_vs_baseline_decisions.csv",
    "receiver_level_signal_manifest.json"
]

missing = []

for filename in expected_outputs:

    path = OUTPUT_DIR / filename

    if path.exists():
        print(f"[PASS] {filename}")
    else:
        print(f"[FAIL] {filename}")
        missing.append(filename)

if missing:
    raise RuntimeError(f"Missing outputs: {missing}")

print("\nNOTEBOOK 08C COMPLETE")
print("Rows:", len(oof))
print("Receivers:", oof.nfl_id.nunique())
print("Outputs:", OUTPUT_DIR)


[PASS] fold_level_model_metrics.csv
[PASS] receiver_level_predictions.csv
[PASS] receiver_level_discrimination.csv
[PASS] receiver_cluster_bootstrap.csv
[PASS] receiver_error_concentration.csv
[PASS] draft_cohort_receiver_metrics.csv
[PASS] model_vs_baseline_decisions.csv
[PASS] receiver_level_signal_manifest.json

NOTEBOOK 08C COMPLETE
Rows: 2232
Receivers: 54
Outputs: D:\PROJECTS\WR-MovementIQ\reports\receiver_level_signal
